# Bronze Layer (Loading the Data Files)

In [0]:
from pyspark.sql.functions import (
    col,
    sum,
    when,
    to_date,
    year,
    month,
    count
)

In [0]:
file_path = "/Volumes/workspace/default/raw_data/Global_Superstore2.csv"
raw_df = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(file_path)
)
display(raw_df)

# Silver Layer (Data Cleaning)

In [0]:
silver_df = raw_df.select(
    col("Row ID").alias("row_id"),
    col("Order ID").alias("order_id"),
    to_date(col("Order Date"), "dd-MM-yyyy").alias("order_date"),
    to_date(col("Ship Date"), "dd-MM-yyyy").alias("ship_date"),
    col("Ship Mode").alias("ship_mode"),
    col("Customer ID").alias("customer_id"),
    col("Customer Name").alias("customer_name"),
    col("Segment").alias("segment"),
    col("City").alias("city"),
    col("State").alias("state"),
    col("Country").alias("country"),
    col("Postal Code").alias("postal_code"),
    col("Market").alias("market"),
    col("Region").alias("region"),
    col("Product ID").alias("product_id"),
    col("Category").alias("category"),
    col("Sub-Category").alias("sub_category"),
    col("Product Name").alias("product_name"),
    col("Sales").cast("double").alias("sales"),
    col("Quantity").cast("int").alias("quantity"),
    col("Discount").cast("double").alias("discount"),
    col("Profit").cast("double").alias("profit"),
    col("Shipping Cost").cast("double").alias("shipping_cost"),
    col("Order Priority").alias("order_priority")
)
display(silver_df)

# Gold Layer (Data Aggregation & Analysis)

In [0]:
gold_region_sales = silver_df.groupBy("region").agg(
    sum("sales").alias("total_sales"),
    sum("profit").alias("total_profit"),
    count("order_id").alias("total_orders")
).orderBy(col("total_sales").desc())
display(gold_region_sales)

In [0]:
gold_top_products = silver_df.groupBy("product_name", "category").agg(
    sum("sales").alias("total_sales"),
    sum("profit").alias("total_profit")
).orderBy(col("total_sales").desc()).limit(10)
display(gold_top_products)